In [2]:
# Python

import pandas as pd
import numpy as np

import requests
from io import BytesIO

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
# Load data

url = "https://raw.githubusercontent.com/Yuliana-wein/Predict-Massive-Transfusion-in-Trauma-PatieUsing-Early-Clinical-Parameters/main/Dataset/CRASH2_Final_Dataset.xlsx"

response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))

df.head()

,Patient_ID,Age,Sex,Systolic_BP_mmHg,Heart_Rate_BPM,Respiratory_Rate_BPM,GCS_Score,Injury_Type,Units_Transfused,Lactate_mmol_L,Arterial_Base_Excess,Time_to_Hospital_min,Shock_Index,MT
0,PT-0001,66,Male,128,61,21,13.0,Blunt,0,1.6,0.6,111,0.48,0
1,PT-0002,43,Male,163,79,15,12.0,Blunt,1,1.2,0.8,75,0.48,0
2,PT-0003,80,Male,141,70,18,11.0,Blunt,1,4.0,-4.7,52,0.50,0
3,PT-0004,37,Male,100,86,10,NaN,Blunt,2,1.6,-4.0,40,0.86,0
4,PT-0005,72,Female,128,74,18,NaN,Blunt,0,0.8,-1.3,64,0.58,0


In [4]:
#------------- Step 1: Check missing values ----------------

print("Missing values before cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
Patient_ID                0
Age                       0
Sex                       0
Systolic_BP_mmHg          0
Heart_Rate_BPM            0
Respiratory_Rate_BPM      0
GCS_Score               379
Injury_Type               0
Units_Transfused          0
Lactate_mmol_L            0
Arterial_Base_Excess      0
Time_to_Hospital_min      0
Shock_Index               0
MT                        0
dtype: int64


In [5]:
# ------  Step 2: Drop irrelevant columns --------

# GCS_Score: 379 missing values + not a key MT predictor → drop
# Patient_ID, Age, Sex, RR, Time_to_Hospital: not clinical MT criteria → drop
# Units_Transfused: used to define MT label, causes data leakage → drop

cols_to_drop = ['Patient_ID', 'Age', 'Sex', 'GCS_Score', 
                'Respiratory_Rate_BPM', 'Time_to_Hospital_min', 
                'Units_Transfused']

df = df.drop(columns=cols_to_drop)
print("Remaining columns:", df.columns.tolist())
print("Shape after dropping:", df.shape)


Remaining columns: ['Systolic_BP_mmHg', 'Heart_Rate_BPM', 'Injury_Type', 'Lactate_mmol_L', 'Arterial_Base_Excess', 'Shock_Index', 'MT']
Shape after dropping: (1000, 7)


In [6]:
# -------- Step 3: Encode categorical feature ---------

# Injury_Type: Penetrating=1, Blunt=0


df['Injury_Type'] = df['Injury_Type'].map({'Penetrating': 1, 'Blunt': 0})
print("Injury_Type encoded:")
print(df['Injury_Type'].value_counts())

Injury_Type encoded:
Injury_Type
0    686
1    314
Name: count, dtype: int64


In [7]:
# ----------- Step 4: Train/ Test

from sklearn.model_selection import train_test_split

# ── Step 5: Train/Test split (80/20) ──
X = df.drop(columns=['MT'])
y = df['MT']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20,      
    random_state=42,     # fixed seed for reproducibility
    stratify=y)          # preserve MT/Non-MT ratio in both sets

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print()
print("Train MT distribution:", y_train.value_counts().to_dict())
print("Test MT distribution:", y_test.value_counts().to_dict())

Train size: (800, 6)
Test size: (200, 6)

Train MT distribution: {0: 600, 1: 200}
Test MT distribution: {0: 150, 1: 50}


In [8]:
# ------------ Step 5: Scale numerical features ─

from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()

# fit_transform on train: scaler learns mean and std from train data
X_train_scaled = scaler.fit_transform(X_train)

# transform only on test: apply same parameters, don't learn new ones
X_test_scaled = scaler.transform(X_test)

print("Scaling done.")
print("X_train_scaled shape:", X_train_scaled.shape)
print()
# Check: after scaling mean should be ~0 and std ~1
print("Mean of first 3 features (train):", X_train_scaled[:, :3].mean(axis=0).round(3))
print("Std of first 3 features (train):", X_train_scaled[:, :3].std(axis=0).round(3))

Scaling done.
X_train_scaled shape: (800, 6)

Mean of first 3 features (train): [-0. -0.  0.]
Std of first 3 features (train): [1. 1. 1.]


In [ ]:
# Save the data 

import pickle

# Save everything including the fitted scaler
with open('preprocessed_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'scaler': scaler
    }, f)

In [ ]:
# Import the dataset

with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
scaler = data['scaler']  # already fitted!